In [10]:
# COMPLETE WORKING SPAM CLASSIFIER
# Copy this entire code and run it in one go

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Step 1: Loading data...")
# Load the dataset
df = pd.read_csv('spam.csv', encoding='latin-1')

# Keep only the first two columns (some versions have extra empty columns)
df = df.iloc[:, :2]
df.columns = ['label', 'message']

print(f"Loaded {len(df)} messages")
print(f"Labels: {df['label'].unique()}")

print("\nStep 2: Preparing data...")
# Convert labels to numbers
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Check if mapping worked
print(f"Sample mapping: {df[['label', 'label_num']].head()}")

# Remove any empty messages
df = df[df['message'].notna()]
df = df[df['message'].str.len() > 0]

print(f"After cleaning: {len(df)} messages")

# Split features and target
X = df['message'].values
y = df['label_num'].values

print(f"Number of spam messages: {y.sum()}")
print(f"Number of ham messages: {len(y) - y.sum()}")

print("\nStep 3: Splitting data...")
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} messages")
print(f"Test set: {len(X_test)} messages")

print("\nStep 4: Converting text to numbers (TF-IDF)...")
# Convert text to numbers
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Features created: {X_train_tfidf.shape[1]}")

print("\nStep 5: Training the model...")
# Train Logistic Regression (works better than Naive Bayes for spam)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("\nStep 6: Making predictions...")
# Predict
y_pred = model.predict(X_test_tfidf)

print("\nStep 7: Results...")
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*50)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['HAM', 'SPAM']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nBreakdown:")
print(f"  True HAM (correctly identified as HAM):  {cm[0,0]}")
print(f"  False SPAM (HAM marked as SPAM by mistake): {cm[0,1]}")
print(f"  False HAM (SPAM marked as HAM by mistake): {cm[1,0]}")
print(f"  True SPAM (correctly identified as SPAM): {cm[1,1]}")

print("\n" + "="*50)
print("TESTING WITH REAL EXAMPLES")
print("="*50)

# Function to test messages
def test_message(text):
    # Convert to numbers using same vectorizer
    text_tfidf = vectorizer.transform([text])
    # Predict
    prediction = model.predict(text_tfidf)[0]
    # Get probability/confidence
    probability = model.predict_proba(text_tfidf)[0]
    
    if prediction == 1:
        confidence = probability[1] * 100
        print(f"🔴 SPAM detected! (Confidence: {confidence:.1f}%)")
        print(f"   Message: {text[:100]}")
    else:
        confidence = probability[0] * 100
        print(f"🟢 NOT SPAM (HAM) (Confidence: {confidence:.1f}%)")
        print(f"   Message: {text[:100]}")
    return prediction

# Test with known spam messages
print("\n📧 Testing with SPAM examples:")
print("-" * 50)
test_message("CONGRATULATIONS! You have won a $1000 Walmart gift card. Click here to claim: http://fake.link")
test_message("URGENT! Your account has been suspended. Verify now at https://fake-bank.com")
test_message("FREE entry to win iPhone 15. Text WIN to 88222 now!")
test_message("You've been selected for a $500 Amazon reward. Claim within 24 hours: bit.ly/fake")

print("\n📧 Testing with HAM (normal) examples:")
print("-" * 50)
test_message("Hey, are we still meeting for coffee tomorrow at 3pm?")
test_message("Don't forget to pick up milk and bread on your way home")
test_message("Thanks for your email. I will review the report and get back to you by Friday.")
test_message("What time does the movie start? Should I buy tickets online?")

print("\n" + "="*50)
print("🎯 TRY YOUR OWN MESSAGE")
print("="*50)

# Let you test your own message
your_message = input("\nType a message to check if it's spam (or press Enter to skip): ")
if your_message.strip():
    test_message(your_message)

print("\n✅ DONE! Model is working correctly.")
print("\n💾 Saving model for future use...")
import joblib
joblib.dump(model, 'spam_classifier.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("✅ Model saved as 'spam_classifier.pkl'")

Step 1: Loading data...
Loaded 5572 messages
Labels: ['ham' 'spam']

Step 2: Preparing data...
Sample mapping:   label  label_num
0   ham          0
1   ham          0
2  spam          1
3   ham          0
4   ham          0
After cleaning: 5572 messages
Number of spam messages: 747
Number of ham messages: 4825

Step 3: Splitting data...
Training set: 4457 messages
Test set: 1115 messages

Step 4: Converting text to numbers (TF-IDF)...
Features created: 5000

Step 5: Training the model...

Step 6: Making predictions...

Step 7: Results...
MODEL PERFORMANCE
Accuracy:  0.9704 (97.04%)
Precision: 1.0000
Recall:    0.7785
F1-Score:  0.8755

Detailed Classification Report:
              precision    recall  f1-score   support

         HAM       0.97      1.00      0.98       966
        SPAM       1.00      0.78      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      11